# 08 – Redundancia, függőségek és kaszkádhatások

**9. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- soros, párhuzamos és k-out-of-n struktúra számítása
- redundancia függetlenségi feltételének kritikája
- common-cause failure hatásának szemléltetése
- függőségi és kaszkád gráf értelmezése

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Redundancia: nem pusztán több komponens

Redundancia csak akkor növeli a rendszer megbízhatóságát a várt mértékben, ha a redundáns elemeket nem ugyanaz a hibaok éri. Közös táp, azonos környezet, shared software vagy közös karbantartási hiba könnyen megsérti a függetlenségi feltevést.


In [ ]:
R=.95
print('2 soros',R**2); print('2 párhuzamos',1-(1-R)**2)


## 2. k-out-of-n

Azonos, független komponenseknél $R_{k|n}=\sum_{i=k}^n {n\choose i}R^i(1-R)^{n-i}$. Példa: 2oo3 voting.


In [ ]:
from math import comb
def k_out_of_n(R,k,n): return sum(comb(n,i)*R**i*(1-R)**(n-i) for i in range(k,n+1))
for r in [.8,.9,.95,.99]: print(r,k_out_of_n(r,2,3))


In [ ]:
Rs=np.linspace(.5,.999,150)
plt.plot(Rs,Rs,label='1oo1'); plt.plot(Rs,[k_out_of_n(r,1,2) for r in Rs],label='1oo2'); plt.plot(Rs,[k_out_of_n(r,2,3) for r in Rs],label='2oo3'); plt.xlabel('component R'); plt.ylabel('system R'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 3. Common-cause failure – egyszerű beta-factor szemléltetés

Nagyon leegyszerűsített oktatási modell: a failure probability $q$ egy $\beta q$ common-cause részből és $(1-\beta)q$ független részből áll. Két elem együttes failure probability-jére: $P_{dual}\approx\beta q+(1-\beta)q^2$. Ez nem teljes CCF standardmodell, csak nagyságrendi szemléltetés.


In [ ]:
q=.01
pd.DataFrame([{'beta':b,'P_dual':b*q+(1-b)*q*q,'ratio_vs_independent':(b*q+(1-b)*q*q)/(q*q)} for b in [0,.01,.05,.1,.2]])


## 4. Függőségi gráf

Nem minden dependency logikai gate. Például common power supply több funkciót érinthet, míg kommunikációs vagy emberi függőség downstream következményeket okozhat.


In [ ]:
import networkx as nx
G=nx.DiGraph([('PowerSupply','LT-101'),('PowerSupply','LSHH-101'),('PowerSupply','XV actuator'),('LT-101','HMI'),('HMI','Operator'),('LSHH-101','TripLogic'),('TripLogic','XV actuator')])
print('PowerSupply downstream:',sorted(nx.descendants(G,'PowerSupply')))
pos=nx.spring_layout(G,seed=3); nx.draw_networkx(G,pos,node_size=1800,font_size=8); plt.axis('off'); plt.show()


## 5. TK-101: common power failure

Az automatikus shutdown független FTA-ja $1-(1-p_{LSHH})(1-p_{XV})$. Ha `COMMON_POWER_FAIL` egyszerre veszélyezteti mindkettőt, azt **egy közös okként** kell reprezentálni, nem két független basic event másolatként.


In [ ]:
p_lshh=.01; p_xv=.005; p_common=.002
ind=1-(1-p_lshh)*(1-p_xv)
with_common=1-(1-p_common)*(1-p_lshh)*(1-p_xv)
print(ind,with_common,'relative increase',with_common/ind-1)


## 6. Kaszkádhatás

Egy strukturális gráf szemléltetheti a propagation útvonalat: SensorBias → LateDetection → HighLevelDuration → OverflowDemand → LossOfContainment → IgnitionOpportunity → FireEscalation. A valószínűségek és időzítés miatt később BN/Markov/szimuláció lehet célszerűbb.


In [ ]:
prop=nx.DiGraph([('SensorBias','LateDetection'),('LateDetection','HighLevelDuration'),('HighLevelDuration','OverflowDemand'),('OverflowDemand','LossOfContainment'),('LossOfContainment','IgnitionOpportunity'),('IgnitionOpportunity','FireEscalation')])
print(list(nx.topological_sort(prop)))


## Próbáld ki!

1. Számíts 2oo3 reliability-t R=0.90 mellett és hasonlíts 1oo2-höz.
2. q=0.001, beta=0.01 mellett hányszoros a dual failure az independence-hez képest?
3. Adj NetworkSwitch common dependency-t a gráfhoz.
4. Azonosíts két TK-101 függőséget, amely inkább szervezeti, mint műszaki.


## Összefoglalás

A redundancia értékét csak a függőségek ismeretében lehet értelmezni. A következő tanegység a rendszerkockázatot már **karbantartási döntési stratégiákhoz** kapcsolja.
